### Unpacking self-attention

<b>Logits:</b> Raw, unnormalised scores generated by the final layer of a neural netowrk before an activation function such as Softmax or Sigmoid

In [6]:
import numpy as np

In [7]:
def softmax(x):
    ## axis = 0 for column-wise, axis = 1 for row-wise, axis = -1 for last dimension
    ## keepdims = True to maintain the same number of dimensions for broadcasting, otherwise dimension will be reduced
    shifted = x - np.max(x, axis = -1, keepdims = True) 
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis = -1, keepdims = True)

logits = np.array([2.0, 1.0, 0.1])
print(f"logits:  {logits}")
print(f"softmax: {softmax(logits)}")
print(f"sum:     {softmax(logits).sum():.4f}")

logits:  [2.  1.  0.1]
softmax: [0.65900114 0.24243297 0.09856589]
sum:     1.0000


* Softmax only cares about the relative differences between the numbers and not the exact values. Hence, there's the shifted = x - max value within the list
* Without this, there will be overflow/ underflow, whereby either all the values are large leading to inf or all the absolute values are small leading to 0 

##### Why substract the maxmimum instead of the minimum value across x
By subtracting the maxmimum value, the exponenet of all values are guarenteed to be under 1 as e**0 = 1 when xi​−max(x)≤0. 

On the other hand, if subtracting the minimum leads to x = 2000, then e**2000 becomes an infinitely large value. Coversely, the x = -2000 may leads to undeflow to zero but is acceptable because thos eterms correspond to probabilities that are alreadyu effectively zero. (Overflow is a bigger issue than underflow in softmax)

#### Sclaed dot product attention 

In [8]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]  # dimension of the key vectors
    scores = Q @ K.T / np.sqrt(d_k) # shape: (seq_len_q, seq_len_k)
    weights = softmax(scores)  # shape: (seq_len_q, seq_len_k)
    output = weights @ V  # shape: (seq_len_q, d_v)
    return output, weights

##### Self-attention class with learned projections

In [9]:
class SelfAttention:
    def __init__(self, d_model, d_k, d_v, seed = 42):
        rng = np.random.default_rng(seed)
        scale = np.sqrt(2.0 / (d_model + d_k))
        self.W_Q = rng.normal(0, scale, (d_model, d_k))
        self.W_K = rng.normal(0, scale, (d_model, d_k))
        scale_v = np.sqrt(2.0 / (d_model + d_v))
        self.W_V = rng.normal(0, scale_v, (d_model, d_v))
        self.d_k = d_k

    def forward(self, X):
        Q = X @ self.W_Q  # shape: (seq_len, d_k)
        K = X @ self.W_K  # shape: (seq_len, d_k)
        V = X @ self.W_V  # shape: (seq_len, d_v)
        output, weights = scaled_dot_product_attention(Q, K, V)
        return output, weights

In [10]:
X

array([[ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472, -1.95103519,
        -1.30217951,  0.1278404 , -0.31624259],
       [-0.01680116, -0.85304393,  0.87939797,  0.77779194,  0.0660307 ,
         1.12724121,  0.46750934, -0.85929246],
       [ 0.36875078, -0.9588826 ,  0.8784503 , -0.04992591, -0.18486236,
        -0.68092954,  1.22254134, -0.15452948],
       [-0.42832782, -0.35213355,  0.53230919,  0.36544406,  0.41273261,
         0.430821  ,  2.1416476 , -0.40641502],
       [-0.51224273, -0.81377273,  0.61597942,  1.12897229, -0.11394746,
        -0.84015648, -0.82448122,  0.65059279],
       [ 0.74325417,  0.54315427, -0.66550971,  0.23216132,  0.11668581,
         0.2186886 ,  0.87142878,  0.22359555]])

In [11]:
sentence = ["The", "cat", "sat", "on", "the", "mat"]
n_tokens = len(sentence)
d_model = 8
dk = 4
dv = 4

rng = np.random.default_rng(42)
X = rng.normal(0, 1, (n_tokens, d_model))

attn = SelfAttention(d_model, dk, dv, seed=42)
output, weights = attn.forward(X)

print("Attention weights (each row: where that token looks):\n")
print(f"{'':>6}", end="")
for token in sentence:
    print(f"{token:>6}", end="")
print()

for i, token in enumerate(sentence):
    print(f"{token:>6}", end="")
    for j in range(n_tokens):
        w = weights[i][j]
        print(f"{w:6.3f}", end="")
    print()

Attention weights (each row: where that token looks):

         The   cat   sat    on   the   mat
   The 0.097 0.122 0.236 0.445 0.047 0.052
   cat 0.188 0.150 0.176 0.144 0.193 0.149
   sat 0.166 0.130 0.213 0.187 0.150 0.154
    on 0.172 0.144 0.146 0.115 0.208 0.215
   the 0.198 0.170 0.214 0.246 0.129 0.043
   mat 0.168 0.152 0.126 0.101 0.220 0.234


In [36]:
def ascii_heatmap(weights, tokens, chars=" ░▒▓█"):
    n = len(tokens)
    print(f"\n{'':>6}", end="")
    for t in tokens:
        print(f"{t:>6}", end="")
    print()

    for i in range(n):
        print(f"{tokens[i]:>6}", end="")
        for j in range(n):
            level = int(weights[i][j] * (len(chars) - 1) / weights.max())
            level = min(level, len(chars) - 1)
            print(f"{'  ' + chars[level] + '   '}", end="")
        print()

ascii_heatmap(weights, sentence)


         The   cat   sat    on   the   mat
   The        ░     ▒     █               
   cat  ░     ░     ░     ░     ░     ░   
   sat  ░     ░     ░     ░     ░     ░   
    on  ░     ░     ░     ░     ░     ░   
   the  ░     ░     ░     ▒     ░         
   mat  ░     ░     ░           ░     ▒   


In [12]:
import torch
import torch.nn as nn

d_model = 8
n_heads = 2
seq_len = 6

mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)

X_torch = torch.randn(1, seq_len, d_model)

output, attn_weights = mha(X_torch, X_torch, X_torch)

print(f"Input shape:            {X_torch.shape}")
print(f"Output shape:           {output.shape}")
print(f"Attention weight shape: {attn_weights.shape}")
print(f"\nAttn weights (averaged over heads):")
print(attn_weights[0].detach().numpy().round(3))

Input shape:            torch.Size([1, 6, 8])
Output shape:           torch.Size([1, 6, 8])
Attention weight shape: torch.Size([1, 6, 6])

Attn weights (averaged over heads):
[[0.131 0.143 0.184 0.152 0.212 0.178]
 [0.108 0.277 0.172 0.149 0.106 0.189]
 [0.135 0.206 0.154 0.171 0.117 0.216]
 [0.128 0.249 0.164 0.163 0.099 0.196]
 [0.412 0.129 0.144 0.118 0.119 0.078]
 [0.226 0.188 0.151 0.173 0.116 0.146]]


##### Exercise 1: Modify scaled_dot_product_attention to accept and optional mask matrix that sets certain postions to negative infinity before softmax (this is how causal/ decoder amsking works)

In [13]:
def scaled_dot_product_attention(Q, K, V, mask = None):
    d_k = Q.shape[-1]  # dimension of the key vectors
    masked_K = K * mask
    scores = Q @ masked_K.T / np.sqrt(d_k) # shape: (seq_len_q, seq_len_k)
    weights = softmax(scores)  # shape: (seq_len_q, seq_len_k)
    output = weights @ V  # shape: (seq_len_q, d_v)
    return output, weights

In [14]:
mask = np.array([1, 1, 1, 1, np.inf, np.inf])

##### Exercise 2: Implement multi-head attention from scratch: split Q, K, V into n_heads chunks, run attention on each, concatenate, and project through a final weight matrix Wo

In [47]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]  # dimension of the key vectors
    scores = Q @ K.T / np.sqrt(d_k) # shape: (seq_len_q, seq_len_k)
    weights = softmax(scores)  # shape: (seq_len_q, seq_len_k)
    output = weights @ V  # shape: (seq_len_q, d_v)
    return output, weights

class MultiHeadAttention:
    def __init__(self, d_model, n_heads, seed = 42): 
        rng = np.random.default_rng(seed)
        self.d_k = d_model // n_heads
        self.d_v = d_model // n_heads
        self.n_heads = n_heads
        self.W_Q = []
        self.W_K = []
        self.W_V = []
        for _ in range(n_heads):
            scale = np.sqrt(2.0 / (d_model + self.d_k))
            self.W_Q.append(rng.normal(0, scale, (d_model, self.d_k)))
            self.W_K.append(rng.normal(0, scale, (d_model, self.d_k)))
            self.W_V.append(rng.normal(0, scale, (d_model, self.d_v)))
        
        self.W_0 = rng.normal(0,
                              np.sqrt(2.0 / (d_model + d_model)),
                              (d_model, d_model))


    def forward(self, X, mask = None):
        output_lst = []
        weights_lst = []
        for i in range(self.n_heads):
            Q = X @ self.W_Q[i]
            K = X @ self.W_K[i]
            V = X @ self.W_V[i]
            output, weights = scaled_dot_product_attention(Q, K, V)
            output_lst.append(output)
            weights_lst.append(weights)
        
        concat_output = np.concatenate(output_lst, axis = 1)
        final_projection = concat_output @ self.W_0 

        return final_projection, np.concatenate(weights_lst, axis = 1), np.stack(weights_lst, axis = 0)

In [48]:
import numpy as np

sentence = ["The", "cat", "sat", "on", "the", "mat"]
n_tokens = len(sentence)
d_model = 8
n_heads = 2
dk = 4
dv = 4

rng = np.random.default_rng(42)
X = rng.normal(0, 1, (n_tokens, d_model))

attn = MultiHeadAttention(d_model, 2, seed=42)
output, weights, w_stack = attn.forward(X)

print("Attention weights (each row: where that token looks):\n")
print(f"{'':>6}", end="")
for token in sentence:
    print(f"{token:>6}", end="")
print()

for i, token in enumerate(sentence):
    print(f"{token:>6}", end="")
    for j in range(n_tokens):
        w = weights[i][j]
        print(f"{w:6.3f}", end="")
    print()

Attention weights (each row: where that token looks):

         The   cat   sat    on   the   mat
   The 0.097 0.122 0.236 0.445 0.047 0.052
   cat 0.188 0.150 0.176 0.144 0.193 0.149
   sat 0.166 0.130 0.213 0.187 0.150 0.154
    on 0.172 0.144 0.146 0.115 0.208 0.215
   the 0.198 0.170 0.214 0.246 0.129 0.043
   mat 0.168 0.152 0.126 0.101 0.220 0.234


In [37]:
output.shape

(6, 8)

In [51]:
weights

array([[0.09738204, 0.12245533, 0.23550689, 0.44533254, 0.04742663,
        0.05189657, 0.13034151, 0.15720097, 0.09997696, 0.18511527,
        0.17007677, 0.25728852],
       [0.18828224, 0.15001171, 0.17610642, 0.14394765, 0.19264035,
        0.14901163, 0.16023592, 0.16464586, 0.21002139, 0.23648624,
        0.08679672, 0.14181387],
       [0.16623127, 0.12985813, 0.21326549, 0.1866732 , 0.14956689,
        0.15440503, 0.11310603, 0.11229861, 0.19313965, 0.28964918,
        0.08194911, 0.20985742],
       [0.17202088, 0.14358728, 0.14587208, 0.115324  , 0.20792098,
        0.2152748 , 0.15022046, 0.10931218, 0.31844676, 0.34877784,
        0.04608993, 0.02715284],
       [0.19834059, 0.16965893, 0.21415862, 0.24608809, 0.12870932,
        0.04304444, 0.1029756 , 0.16013671, 0.04627676, 0.06137366,
        0.19304756, 0.43618971],
       [0.16774733, 0.1516455 , 0.12556837, 0.1006086 , 0.22020835,
        0.23422185, 0.18949668, 0.1491144 , 0.2976416 , 0.21017486,
        0.09543295,

In [49]:
w_stack

array([[[0.09738204, 0.12245533, 0.23550689, 0.44533254, 0.04742663,
         0.05189657],
        [0.18828224, 0.15001171, 0.17610642, 0.14394765, 0.19264035,
         0.14901163],
        [0.16623127, 0.12985813, 0.21326549, 0.1866732 , 0.14956689,
         0.15440503],
        [0.17202088, 0.14358728, 0.14587208, 0.115324  , 0.20792098,
         0.2152748 ],
        [0.19834059, 0.16965893, 0.21415862, 0.24608809, 0.12870932,
         0.04304444],
        [0.16774733, 0.1516455 , 0.12556837, 0.1006086 , 0.22020835,
         0.23422185]],

       [[0.13034151, 0.15720097, 0.09997696, 0.18511527, 0.17007677,
         0.25728852],
        [0.16023592, 0.16464586, 0.21002139, 0.23648624, 0.08679672,
         0.14181387],
        [0.11310603, 0.11229861, 0.19313965, 0.28964918, 0.08194911,
         0.20985742],
        [0.15022046, 0.10931218, 0.31844676, 0.34877784, 0.04608993,
         0.02715284],
        [0.1029756 , 0.16013671, 0.04627676, 0.06137366, 0.19304756,
         0.43618971